# 滞后特征工程 - 最简化版本

**目标：** 为训练数据添加前一天的responder值作为特征

**为什么要这样做？**
- 时间序列数据有自相关性，昨天的值对预测今天有帮助
- Kaggle API在推理时会提供前一天的lags，训练时也需要模拟

## 1. 环境设置

In [1]:
import polars as pl
import gc
from pathlib import Path
from tqdm import tqdm

# 路径设置
DATA_PATH = Path('../Dataset')
OUTPUT_PATH = Path('./processed_data')
OUTPUT_PATH.mkdir(exist_ok=True, parents=True)

print(f"数据路径: {DATA_PATH}")
print(f"输出路径: {OUTPUT_PATH}")

数据路径: ../Dataset
输出路径: processed_data


## 2. 滞后特征创建函数

**核心逻辑：**
1. 对每个股票、每天，找到最后时间点的responder值
2. 向后shift一天，得到前一天的值
3. 把这些值作为特征join回原数据

In [2]:
def add_lag_features(df):
    """
    为数据添加滞后特征
    
    输入: 原始数据（含responder_0到responder_8）
    输出: 添加了responder_X_lag_1列的数据
    """
    responder_cols = [f'responder_{i}' for i in range(9)]
    existing_responders = [col for col in responder_cols if col in df.columns]
    
    # 步骤1: 获取每天每个股票最后时间点的responder值
    last_values = (
        df.sort(['symbol_id', 'date_id', 'time_id'])
        .group_by(['symbol_id', 'date_id'])
        .agg([
            pl.col('time_id').last().alias('last_time_id'),
            *[pl.col(col).last().alias(f'{col}_last') for col in existing_responders]
        ])
    )
    
    # 步骤2: 向前shift一天，得到前一日的值
    last_values_with_lag = (
        last_values
        .sort(['symbol_id', 'date_id'])
        .with_columns([
            pl.col(f'{col}_last').shift(1).over('symbol_id').alias(f'{col}_lag_1')
            for col in existing_responders
        ])
    )
    
    # 步骤3: 只保留需要的列
    lag_cols = ['symbol_id', 'date_id'] + [f'{col}_lag_1' for col in existing_responders]
    last_values_with_lag = last_values_with_lag.select(lag_cols)
    
    # 步骤4: 将滞后值join回原数据
    df_with_lags = df.join(
        last_values_with_lag,
        on=['symbol_id', 'date_id'],
        how='left'
    )
    
    return df_with_lags

print("滞后特征函数已定义")

滞后特征函数已定义


## 3. 处理所有分区并保存

**流程：**
1. 逐个分区处理，添加lag特征
2. 合并所有数据
3. 删除当天的responder列（除了responder_6，因为它是目标）
4. 分割训练集和验证集
5. 保存

In [3]:
def process_all_data(
    data_path=DATA_PATH,
    output_path=OUTPUT_PATH,
    skip_dates=500,          # 跳过前N天
    validation_dates=100     # 最后N天作为验证集
):
    """
    处理所有数据：添加lag特征 + 删除当天responder + 分割保存
    """
    
    # 获取所有分区
    train_path = data_path / 'train.parquet'
    partitions = sorted([
        int(d.name.split('=')[1]) 
        for d in train_path.iterdir() 
        if d.is_dir()
    ])
    
    print(f"找到 {len(partitions)} 个分区: {partitions}")
    
    # ========== 第一步：处理所有分区 ==========
    all_data = []
    
    for pid in tqdm(partitions, desc="添加lag特征"):
        file_path = train_path / f'partition_id={pid}/part-0.parquet'
        df = pl.read_parquet(file_path)
        df_with_lags = add_lag_features(df)
        all_data.append(df_with_lags)
        del df, df_with_lags
        gc.collect()
    
    # ========== 第二步：合并所有数据 ==========
    print("\n合并所有数据...")
    combined = pl.concat(all_data)
    
    # 转换为float32节省空间
    for col in combined.columns:
        if combined[col].dtype == pl.Float64:
            combined = combined.with_columns(pl.col(col).cast(pl.Float32))
    
    print(f"合并后形状: {combined.shape}")
    print(f"日期范围: {combined['date_id'].min()} 到 {combined['date_id'].max()}")
    
    del all_data
    gc.collect()
    
    # ========== 第三步：删除当天的responder列 ==========
    print("\n删除当天的responder列（保留responder_6作为目标）...")
    
    # 要删除的列：responder_0,1,2,3,4,5,7,8（不删除responder_6）
    cols_to_drop = [f'responder_{i}' for i in range(9) if i != 6]
    
    # 检查哪些列存在
    existing_cols_to_drop = [col for col in cols_to_drop if col in combined.columns]
    
    if existing_cols_to_drop:
        combined = combined.drop(existing_cols_to_drop)
        print(f"已删除: {existing_cols_to_drop}")
        print(f"删除后形状: {combined.shape}")
    
    # ========== 第四步：过滤和分割数据 ==========
    print(f"\n过滤和分割数据...")
    print(f"  跳过date_id < {skip_dates}")
    print(f"  最后{validation_dates}天作为验证集")
    
    # 过滤
    combined = combined.filter(pl.col('date_id') >= skip_dates)
    
    # 分割
    all_dates = sorted(combined['date_id'].unique())
    train_dates = all_dates[:-validation_dates]
    valid_dates = all_dates[-validation_dates:]
    
    print(f"  训练集: {train_dates[0]}-{train_dates[-1]} ({len(train_dates)}天)")
    print(f"  验证集: {valid_dates[0]}-{valid_dates[-1]} ({len(valid_dates)}天)")
    
    train_data = combined.filter(pl.col('date_id').is_in(train_dates))
    valid_data = combined.filter(pl.col('date_id').is_in(valid_dates))
    
    print(f"  训练数据: {train_data.shape}")
    print(f"  验证数据: {valid_data.shape}")
    
    # ========== 第五步：保存 ==========
    print("\n保存数据...")
    
    # 修复：DataFrame.write_parquet(路径)
    train_data.write_parquet(output_path / 'training.parquet')
    print("✓ training.parquet")
    
    valid_data.write_parquet(output_path / 'validation.parquet')
    print("✓ validation.parquet")
    
    # ========== 完成 ==========
    print("\n" + "="*50)
    print("处理完成!")
    print("="*50)
    print(f"\n最终列数: {len(train_data.columns)}")
    print(f"\n列名:")
    print(train_data.columns)

# 运行处理
process_all_data()

找到 10 个分区: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


添加lag特征: 100%|██████████| 10/10 [00:29<00:00,  2.91s/it]



合并所有数据...
合并后形状: (47127338, 101)
日期范围: 0 到 1698

删除当天的responder列（保留responder_6作为目标）...
已删除: ['responder_0', 'responder_1', 'responder_2', 'responder_3', 'responder_4', 'responder_5', 'responder_7', 'responder_8']
删除后形状: (47127338, 93)

过滤和分割数据...
  跳过date_id < 500
  最后100天作为验证集
  训练集: 500-1598 (1099天)
  验证集: 1599-1698 (100天)
  训练数据: (35861029, 93)
  验证数据: (3716152, 93)

保存数据...
✓ training.parquet
✓ validation.parquet

处理完成!

最终列数: 93

列名:
['date_id', 'time_id', 'symbol_id', 'weight', 'feature_00', 'feature_01', 'feature_02', 'feature_03', 'feature_04', 'feature_05', 'feature_06', 'feature_07', 'feature_08', 'feature_09', 'feature_10', 'feature_11', 'feature_12', 'feature_13', 'feature_14', 'feature_15', 'feature_16', 'feature_17', 'feature_18', 'feature_19', 'feature_20', 'feature_21', 'feature_22', 'feature_23', 'feature_24', 'feature_25', 'feature_26', 'feature_27', 'feature_28', 'feature_29', 'feature_30', 'feature_31', 'feature_32', 'feature_33', 'feature_34', 'feature_35', 'featu

## 4. 验证结果

In [4]:
# 快速验证处理结果
training_file = OUTPUT_PATH / 'training.parquet'
validation_file = OUTPUT_PATH / 'validation.parquet'

if training_file.exists():
    train = pl.read_parquet(training_file)
    print("=== 训练数据 ===")
    print(f"形状: {train.shape}")
    print(f"\n列名 ({len(train.columns)}列):")
    print(train.columns)
    
    # 检查lag特征
    lag_cols = [c for c in train.columns if 'lag' in c]
    print(f"\nLag特征列 ({len(lag_cols)}列):")
    print(lag_cols)
    
    # 检查是否还有当天的responder（除了responder_6）
    responder_cols = [c for c in train.columns if c.startswith('responder_') and 'lag' not in c]
    print(f"\n当天的responder列 ({len(responder_cols)}列):")
    print(responder_cols)
    
    # 查看样本
    sample = train.select(['date_id', 'symbol_id', 'responder_6', 'responder_6_lag_1']).head(5)
    print("\n样本数据:")
    print(sample)

if validation_file.exists():
    valid = pl.read_parquet(validation_file)
    print("\n=== 验证数据 ===")
    print(f"形状: {valid.shape}")
    print(f"日期范围: {valid['date_id'].min()} 到 {valid['date_id'].max()}")

=== 训练数据 ===
形状: (35861029, 93)

列名 (93列):
['date_id', 'time_id', 'symbol_id', 'weight', 'feature_00', 'feature_01', 'feature_02', 'feature_03', 'feature_04', 'feature_05', 'feature_06', 'feature_07', 'feature_08', 'feature_09', 'feature_10', 'feature_11', 'feature_12', 'feature_13', 'feature_14', 'feature_15', 'feature_16', 'feature_17', 'feature_18', 'feature_19', 'feature_20', 'feature_21', 'feature_22', 'feature_23', 'feature_24', 'feature_25', 'feature_26', 'feature_27', 'feature_28', 'feature_29', 'feature_30', 'feature_31', 'feature_32', 'feature_33', 'feature_34', 'feature_35', 'feature_36', 'feature_37', 'feature_38', 'feature_39', 'feature_40', 'feature_41', 'feature_42', 'feature_43', 'feature_44', 'feature_45', 'feature_46', 'feature_47', 'feature_48', 'feature_49', 'feature_50', 'feature_51', 'feature_52', 'feature_53', 'feature_54', 'feature_55', 'feature_56', 'feature_57', 'feature_58', 'feature_59', 'feature_60', 'feature_61', 'feature_62', 'feature_63', 'feature_64', '

## 5. 总结

**处理流程：**
```
原始数据
  ↓
添加lag特征 (responder_X_lag_1 = 前一天的值)
  ↓
删除当天responder (保留responder_6作为目标)
  ↓
分割训练集/验证集
  ↓
保存
```

**最终数据列：**
- `feature_00` ~ `feature_78` (79列) - 原始特征
- `responder_6_lag_1` 等 (9列) - 前一天的responder值
- `responder_6` (1列) - 当天的目标值
- 其他列：date_id, time_id, symbol_id, weight等

**内存节省：** 删除了8个responder列，约占9%内存